In [1]:
import time
import random

# --- 1. THE ENVIRONMENT ---
class VacuumWorld:
    def __init__(self):
        # Locations A and B: 1 is Dirty, 0 is Clean
        self.locations = {'A': random.randint(0, 1), 'B': random.randint(0, 1)}
        self.agent_pos = random.choice(['A', 'B'])
        self.performance_score = 0

    def get_percept(self):
        return (self.agent_pos, self.locations[self.agent_pos])

    def execute_action(self, action):
        if action == 'Suck':
            self.locations[self.agent_pos] = 0
            self.performance_score += 10 # Reward for cleaning
            return "Cleaning..."
        elif action == 'Right':
            self.agent_pos = 'B'
            self.performance_score -= 1 # Cost for moving
            return "Moving to B"
        elif action == 'Left':
            self.agent_pos = 'A'
            self.performance_score -= 1 # Cost for moving
            return "Moving to A"
        return "NoOp (Idle)"

# --- 2. SIMPLE REFLEX AGENT (No Memory) ---
class SimpleReflexAgent:
    def act(self, percept):
        location, status = percept
        if status == 1:
            return 'Suck'
        return 'Right' if location == 'A' else 'Left'

# --- 3. MODEL-BASED AGENT (With Internal State) ---
class ModelBasedAgent:
    def __init__(self):
        # The 'Model' tracks what the agent believes the world looks like
        self.model = {'A': None, 'B': None}

    def act(self, percept):
        location, status = percept
        self.model[location] = status # Update internal memory

        # Logic: If current spot is dirty, clean it.
        if status == 1:
            return 'Suck'

        # Logic: If both spots are clean in the model, STOP (NoOp)
        if self.model['A'] == 0 and self.model['B'] == 0:
            return 'NoOp'

        # Otherwise, move to the other room
        return 'Right' if location == 'A' else 'Left'

# --- 4. THE SIMULATION RUNNER ---
def run_simulation(agent_type):
    env = VacuumWorld()
    agent = SimpleReflexAgent() if agent_type == "Reflex" else ModelBasedAgent()

    print(f"\n--- Starting {agent_type} Agent Simulation ---")
    print(f"Initial State: {env.locations} | Agent starts at: {env.agent_pos}")

    for step in range(1, 6): # Run for 5 steps
        percept = env.get_percept()
        action = agent.act(percept)
        result = env.execute_action(action)

        print(f"Step {step}: Loc: {percept[0]} | Status: {'Dirty' if percept[1] else 'Clean'} | Action: {action} ({result})")

        if action == 'NoOp':
            print("Target achieved. Agent stopped.")
            break

    print(f"Final Performance Score: {env.performance_score}")

# Run Both
run_simulation("Reflex")
run_simulation("Model-Based")


--- Starting Reflex Agent Simulation ---
Initial State: {'A': 0, 'B': 1} | Agent starts at: B
Step 1: Loc: B | Status: Dirty | Action: Suck (Cleaning...)
Step 2: Loc: B | Status: Clean | Action: Left (Moving to A)
Step 3: Loc: A | Status: Clean | Action: Right (Moving to B)
Step 4: Loc: B | Status: Clean | Action: Left (Moving to A)
Step 5: Loc: A | Status: Clean | Action: Right (Moving to B)
Final Performance Score: 6

--- Starting Model-Based Agent Simulation ---
Initial State: {'A': 0, 'B': 0} | Agent starts at: B
Step 1: Loc: B | Status: Clean | Action: Left (Moving to A)
Step 2: Loc: A | Status: Clean | Action: NoOp (NoOp (Idle))
Target achieved. Agent stopped.
Final Performance Score: -1


In [2]:
# --- 4. GOAL-BASED AGENT (With a Target Destination) ---
class GoalBasedAgent:
    def __init__(self, dock_location='A'):
        self.model = {'A': None, 'B': None}
        self.goal_location = dock_location # The "Charging Dock"

    def act(self, percept):
        location, status = percept
        self.model[location] = status

        # 1. If current spot is dirty, clean it (Priority 1)
        if status == 1:
            return 'Suck'

        # 2. Check if the whole world is clean based on our memory
        all_clean = (self.model['A'] == 0 and self.model['B'] == 0)

        # 3. If everything is clean AND we are at the Dock, then STOP
        if all_clean and location == self.goal_location:
            return 'NoOp'

        # 4. If everything is clean but we AREN'T at the dock, move towards it
        if all_clean and location != self.goal_location:
            return 'Left' if self.goal_location == 'A' else 'Right'

        # 5. If we don't know if the other room is clean, go check it
        return 'Right' if location == 'A' else 'Left'

# --- Run the Goal-Based Simulation ---
def run_goal_sim():
    env = VacuumWorld()
    # Force a "Dirty" start to see the full path
    env.locations = {'A': 0, 'B': 1}
    env.agent_pos = 'B'

    agent = GoalBasedAgent(dock_location='A')

    print(f"\n--- Starting Goal-Based Agent (Goal: Clean + Dock at A) ---")

    for step in range(1, 7):
        percept = env.get_percept()
        action = agent.act(percept)
        result = env.execute_action(action)
        print(f"Step {step}: Loc: {percept[0]} | Action: {action} | Score: {env.performance_score}")

        if action == 'NoOp':
            print("Successfully Cleaned and Docked!")
            break

run_goal_sim()


--- Starting Goal-Based Agent (Goal: Clean + Dock at A) ---
Step 1: Loc: B | Action: Suck | Score: 10
Step 2: Loc: B | Action: Left | Score: 9
Step 3: Loc: A | Action: NoOp | Score: 9
Successfully Cleaned and Docked!


In [4]:
import time
import random
from IPython.display import clear_output

class ColabVacuumWorld:
    def __init__(self):
        # 0: Clean, 1: Dirty
        self.locations = {'A': random.randint(0, 1), 'B': random.randint(0, 1)}
        self.agent_pos = random.choice(['A', 'B'])
        self.score = 0

    def draw_world(self, step, action):
        clear_output(wait=True) # Clears the Colab cell output

        print(f"=== STEP {step} ===")
        print(f"Current Action: {action}")
        print(f"Performance Score: {self.score}")
        print("\n" + "="*30)

        # Room A display
        room_a = "🧹" if self.agent_pos == 'A' else "  "
        dirt_a = "💩" if self.locations['A'] == 1 else "✨"

        # Room B display
        room_b = "🧹" if self.agent_pos == 'B' else "  "
        dirt_b = "💩" if self.locations['B'] == 1 else "✨"

        print(f"|  Room A: {dirt_a} {room_a}  |  Room B: {dirt_b} {room_b}  |")
        print("="*30)
        print("Legend: 🧹 Vacuum | 💩 Dirt | ✨ Clean")

    def execute_action(self, action):
        if action == 'Suck':
            self.locations[self.agent_pos] = 0
            self.score += 10
        elif action == 'Right':
            self.agent_pos = 'B'
            self.score -= 1
        elif action == 'Left':
            self.agent_pos = 'A'
            self.score -= 1

class GoalBasedAgent:
    def __init__(self, dock='A'):
        self.model = {'A': None, 'B': None}
        self.dock = dock

    def act(self, loc, status):
        self.model[loc] = status
        # Priority 1: Clean if dirty
        if status == 1: return 'Suck'
        # Priority 2: If everything is clean, go to dock
        if self.model['A'] == 0 and self.model['B'] == 0:
            if loc == self.dock: return 'NoOp'
            return 'Left' if self.dock == 'A' else 'Right'
        # Priority 3: Explore the other room
        return 'Right' if loc == 'A' else 'Left'

# --- Execution Loop ---
env = ColabVacuumWorld()
agent = GoalBasedAgent(dock='A')
action = "Initializing..."

for i in range(1, 10):
    loc, status = env.agent_pos, env.locations[env.agent_pos]
    env.draw_world(i, action)

    action = agent.act(loc, status)
    if action == 'NoOp':
        env.draw_world(i, "DOCKED & FINISHED ✅")
        break

    time.sleep(1.2) # Delay to watch the animation
    env.execute_action(action)

=== STEP 4 ===
Current Action: DOCKED & FINISHED ✅
Performance Score: 8

|  Room A: ✨ 🧹  |  Room B: ✨     |
Legend: 🧹 Vacuum | 💩 Dirt | ✨ Clean
